Utilizzate il dataset CIFAR-10 (immagini a colori 32x32), crea una pipeline che:
- carichi i dati integrati di keras
- utilizzi una divisione automatica del 15% per la validazione
- inserisce un layer di normalizzazione che porti i dati nell'intervallo [-1,1] invece di [0,1]
Suggerimento: la formula per mappare [0,255] in [-1,1] richiede di riscalare per 1/127.5 e poi sottrarre 1

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models, datasets

#1. CARICAMENTO ASINCRONO E DIVISIONE AUTOMATICA DEI DATI (punto chieve 1 e 3)
#Utilizziamo MIST come esempio di dataset
print("Caricamento dataset in corso...")
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

#Creazione di un oggetto Dataset per gestire la pipeline
#Usiamo il 15% del training per la validazione internamento durante il fit
#il metodo shuffle e batch creano la pipeline di caricamento asincrono
train_ds=tf.data.Dataset.from_tensor_slices((train_images,train_labels))
train_ds=train_ds.shuffle(10000).batch(64).prefetch(tf.data.AUTOTUNE)
#from_tensor_slices: crea un dataset a partire da array numpy dividendo i dati in coppie (immagine, etichetta)
#mescola i dati e li carica in batch di 64
#prefetch: prepara il batch successivo mentre il modello sta ancora lavorando sul batch corrente in GPU

#2. COSTRUZIONE DEL MODELLO E NORMALIZZAZIONE INTEGRATA (punto chieve 2)
print("Costruzione del modello in corso...")
model = models.Sequential([
    # Layer di riscalamento per mappare [0, 255] nell'intervallo [-1, 1]
    # Formula: (input * 1/127.5) - 1
    # Esempio: 255 * (1/127.5) - 1 = 2 - 1 = 1
    # Esempio: 0 * (1/127.5) - 1 = -1
    layers.Rescaling(1./127.5, offset=-1, input_shape=(32, 32, 3)), #normalizzazione integrata

    layers.Flatten(), # appiattisce l'immagine 32x32 in un vettore di 1024 elementi
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

#3. COMPILAZIONE DEL MODELLO E DIVISIONE AUTOMATICA DEI DATI(punto chieve 4)
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy']) 

#Eseguiamo l'addestramento usanto il validation_split automatico di Keras
#Nota: validation_split funziona direttamente sui array numpy
print("Inizio addestramento...")
history=model.fit(
    train_images
    ,train_labels
    , epochs=5
    , validation_split=0.15 #divisione automatica 85% 15%
    , shuffle=True
    , batch_size=64) 

print("Addestramento completato.")

# Verifica finale sui dati di test
print("\nValutazione sul Test Set:")
model.evaluate(test_images, test_labels)

Caricamento dataset in corso...
Costruzione del modello in corso...
Inizio addestramento...


c:\Users\uberti\.conda\envs\ai_epicode\Lib\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
665/665 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.4001 - loss: 1.7074 - val_accuracy: 0.4404 - val_loss: 1.5901
Epoch 2/5
665/665 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.4792 - loss: 1.4815 - val_accuracy: 0.4603 - val_loss: 1.5321
Epoch 3/5
665/665 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.5170 - loss: 1.3711 - val_accuracy: 0.4905 - val_loss: 1.4545
Epoch 4/5
665/665 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.5479 - loss: 1.2876 - val_accuracy: 0.5053 - val_loss: 1.4505
Epoch 5/5
665/665 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.5716 - loss: 1.2131 - val_accuracy: 0.5071 - val_loss: 1.4389
Addestramento completato.

Valutazione sul Test Set:
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5056 - loss: 1.4222


[1.4222393035888672, 0.5055999755859375]